#  HT_05 — Triage Analysis

Categorize SAW scores into P1/P2/P3 and analyze the results.

*Related: [Methodology](../docs/methodology.md)*


In [ ]:
# Repo root on sys.path so `src` imports work
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.config import THRESHOLDS, TRIAGE_ACTIONS, TRIAGE_LABELS
from src.data.cleaner import clean
from src.data.loader import load_raw
from src.data.transformer import transform
from src.dss.saw import calculate_score, normalize
from src.dss.triage import classify_triage


In [ ]:
df = transform(clean(load_raw('../data/raw/cleve.mod')))
from src.config import BENEFIT, CRITERIA, WEIGHTS
norm = normalize(df, CRITERIA, BENEFIT)
df['score'] = calculate_score(norm, WEIGHTS).round(4)
df['triage'] = classify_triage(df['score'], THRESHOLDS, TRIAGE_LABELS)
print('pipeline ready')

## 1. Triage Distribution


In [ ]:
dist = df['triage'].value_counts().reset_index()
dist.columns = ['triage', 'count']
dist['percentage'] = (dist['count'] / dist['count'].sum() * 100).round(1)
dist['action'] = dist['triage'].str[:2].map(TRIAGE_ACTIONS)
dist

## 2. Triage vs Clinical Diagnosis

Crosstab of triage categories against the dataset's own diagnosis label.


In [ ]:
pd.crosstab(df['triage'], df['class'], margins=True)

## 3. Threshold Sensitivity

How would the P1 share change with different thresholds?


In [ ]:
grid = [0.55, 0.60, 0.65, 0.70, 0.75]
pd.DataFrame({
    'P1_threshold': grid,
    'P1_count': [int((df['score'] >= t).sum()) for t in grid],
    'P1_pct': [round((df['score'] >= t).mean() * 100, 1) for t in grid],
})

## 4. Criteria Profile per Triage Level


In [ ]:
df.groupby('triage')[CRITERIA].mean().round(2).T

## 5. Save Final Results


In [ ]:
df.to_csv('../data/processed/cleve_triage.csv', index=False)
print('saved -> data/processed/cleve_triage.csv')

## Summary

- Triage distribution reported with actions
- Crosstab against `class` validates plausibility
- Thresholds recalibratable via `src/config.py`
